# Research 42 — Fixed VIP × region + ExtraTrees complementarity blend

**Hypothesis:** ExtraTrees has shown a modest fixed 80/20 uplift across two partitions and may contribute complementary errors to the four-seed VIP × region leader. Test one fixed 80/10/10 mean of the current reference, four-seed VIP × region component mean, and two-seed ExtraTrees component mean. Weights are fixed; no search or Test access. Report fold-slice scores and a paired class-stratified bootstrap as a conditional diagnostic only.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold

ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "data/train.csv").is_file()
)
y = pd.read_csv(ROOT / "data/train.csv", usecols=["segment"]).segment.to_numpy(dtype=int)


def load_probs(filename):
    frame = pd.read_csv(ROOT / "predictions" / filename)
    assert len(frame) == len(y) and np.array_equal(frame.segment.to_numpy(), y), filename
    probs = frame[[f"prob_{i}" for i in range(4)]].to_numpy(dtype=float)
    assert np.isfinite(probs).all() and (probs >= 0).all(), filename
    sums = probs.sum(axis=1, keepdims=True)
    assert (sums > 0).all(), filename
    return probs / sums


p_base = load_probs("segment-oof.csv")
vip_seed_files = [
    "segment-vip-region-balanced-oof.csv",
    "segment-vip-region-seed-2025-oof.csv",
    "segment-vip-region-seed-1337-oof.csv",
    "segment-vip-region-seed-31415-oof.csv",
]
vip_probs = [load_probs(filename) for filename in vip_seed_files]
p_vip_four = np.mean(vip_probs, axis=0)
p_extra = np.mean(
    [
        load_probs("segment-lightgbm-extra-trees-vip-region-oof.csv"),
        load_probs("segment-lightgbm-extra-trees-vip-region-seed-2025-oof.csv"),
    ],
    axis=0,
)
p_current = 0.8 * p_base + 0.2 * p_vip_four
p_threeway = 0.8 * p_base + 0.1 * p_vip_four + 0.1 * p_extra
p_saved_current = load_probs("segment-vip-region-four-seed-average-oof.csv")
assert np.allclose(p_current, p_saved_current, atol=1e-12)
candidates = {
    "Current reference": p_base,
    "Current 80/20 four-seed leader": p_current,
    "Fixed 80/10/10 VIP4 + ExtraTrees2": p_threeway,
}
fold_seeds = [42, 2025, 1337, 31415]
fold_sets = {
    seed: list(
        StratifiedKFold(n_splits=3, shuffle=True, random_state=seed).split(np.zeros(len(y)), y)
    )
    for seed in fold_seeds
}
rows = []
for name, probs in candidates.items():
    pred = probs.argmax(axis=1)
    row = {
        "candidate": name,
        "macro_f1": f1_score(y, pred, average="macro"),
        "weighted_f1": f1_score(y, pred, average="weighted"),
    }
    for seed, folds in fold_sets.items():
        for fold, (_, valid_idx) in enumerate(folds, start=1):
            row[f"seed{seed}_fold{fold}_macro_f1"] = f1_score(
                y[valid_idx], pred[valid_idx], average="macro"
            )
    rows.append(row)
results = pd.DataFrame(rows)
display(results.round(6))
results.to_csv(ROOT / "assets/segment-vip-region-extratrees-threeway.csv", index=False)
pd.DataFrame(p_threeway, columns=[f"prob_{i}" for i in range(4)]).assign(segment=y).to_csv(
    ROOT / "predictions/segment-vip-region-extratrees-threeway-oof.csv", index=False
)
print("Fixed 80/10/10 OOF blend computed; no model fitting or Test access.")

,candidate,macro_f1,weighted_f1,seed42_fold1_macro_f1,seed42_fold2_macro_f1,seed42_fold3_macro_f1,seed2025_fold1_macro_f1,seed2025_fold2_macro_f1,seed2025_fold3_macro_f1,seed1337_fold1_macro_f1,seed1337_fold2_macro_f1,seed1337_fold3_macro_f1,seed31415_fold1_macro_f1,seed31415_fold2_macro_f1,seed31415_fold3_macro_f1
0,Current reference,0.750895,0.764683,0.749445,0.749821,0.753390,0.750797,0.749412,0.752462,0.749956,0.753540,0.749163,0.749806,0.750854,0.752005
1,Current 80/20 four-seed leader,0.752172,0.765704,0.750552,0.751365,0.754568,0.751751,0.751072,0.753684,0.752052,0.754397,0.750051,0.751079,0.752230,0.753192
2,Fixed 80/10/10 VIP4 + ExtraTrees2,0.751994,0.765510,0.750364,0.751171,0.754415,0.751692,0.751056,0.753222,0.751902,0.754319,0.749744,0.751055,0.751982,0.752934


Fixed 80/10/10 OOF blend computed; no model fitting or Test access.


In [2]:
def macro_f1_from_counts(sample_y, sample_pred):
    confusion = np.bincount(sample_y * 4 + sample_pred, minlength=16).reshape(4, 4)
    true_positive = np.diag(confusion)
    denominator = confusion.sum(axis=0) + confusion.sum(axis=1)
    class_f1 = np.divide(
        2 * true_positive, denominator, out=np.zeros(4, dtype=float), where=denominator > 0
    )
    return class_f1.mean()


predictions = {name: probs.argmax(axis=1) for name, probs in candidates.items()}
class_indices = [np.flatnonzero(y == label) for label in np.unique(y)]
rng = np.random.default_rng(42)
n_bootstrap = 300
comparisons = [
    ("threeway minus reference", "Fixed 80/10/10 VIP4 + ExtraTrees2", "Current reference"),
    (
        "threeway minus four-seed leader",
        "Fixed 80/10/10 VIP4 + ExtraTrees2",
        "Current 80/20 four-seed leader",
    ),
]
bootstrap_deltas = {name: np.empty(n_bootstrap) for name, _, _ in comparisons}
for sample_number in range(n_bootstrap):
    sample = np.concatenate(
        [rng.choice(indices, size=len(indices), replace=True) for indices in class_indices]
    )
    sample_y = y[sample]
    scores = {
        name: macro_f1_from_counts(sample_y, predictions[name][sample]) for name in candidates
    }
    for comparison, candidate, baseline in comparisons:
        bootstrap_deltas[comparison][sample_number] = scores[candidate] - scores[baseline]
boot_rows = []
for name, values in bootstrap_deltas.items():
    boot_rows.append(
        {
            "comparison": name,
            "mean_delta": values.mean(),
            "delta_ci_2_5pct": np.quantile(values, 0.025),
            "delta_ci_97_5pct": np.quantile(values, 0.975),
            "fraction_positive": (values > 0).mean(),
        }
    )
bootstrap_results = pd.DataFrame(boot_rows)
display(bootstrap_results.round(6))
bootstrap_results.to_csv(
    ROOT / "assets/segment-vip-region-extratrees-threeway-bootstrap.csv", index=False
)
print(
    f"Completed {n_bootstrap} paired stratified resamples. Conditional on selected OOF predictions; no Test access."
)

,comparison,mean_delta,delta_ci_2_5pct,delta_ci_97_5pct,fraction_positive
0,threeway minus reference,0.001074,0.000287,0.001790,0.996667
1,threeway minus four-seed leader,-0.000174,-0.000632,0.000221,0.200000


Completed 300 paired stratified resamples. Conditional on selected OOF predictions; no Test access.


## Interpretation limits

The fixed mixture is a single complementarity screen, not a search. Fold slices and bootstrap resampling reuse the same OOF predictions and training labels; the bootstrap does not include fold, training, or repeated-selection uncertainty. Even if it improves the four-seed leader, retain it only as a screening result until genuinely independent validation is available.